# 02 · Reference fingerprint
Tiny Aya Global's refusal-vector fingerprint on split A, then on split B, compared (stability check).
GPU runtime (T4). Logic in `src/forged/fingerprint.py`.

**Result so far:** cosine A vs B = 0.9953, every layer 10–27 at 0.99 or above.

In [ ]:
# ---- Setup: run first, in every new Colab session ----
# Colab secrets needed (key icon, Notebook access ON): HF_TOKEN (Hugging Face, write to team-doom)
# and GH_TOKEN (GitHub, read access to mathcrypto/forged-fingerprints while the repo is private).
import os, sys, subprocess
from google.colab import userdata

HF_ORG = 'team-doom'
REPO = 'mathcrypto/forged-fingerprints'
REPO_DIR = '/content/forged-fingerprints'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://{userdata.get("GH_TOKEN")}@github.com/{REPO}.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]'], check=True)

# Make the package importable now (no restart needed) and visible to !python scripts
sys.path.insert(0, f'{REPO_DIR}/src')
os.environ['PYTHONPATH'] = f'{REPO_DIR}/src'

from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
import forged
print('forged', forged.__version__, '| HF user:', whoami()['name'], '| cwd:', os.getcwd())

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(f'{HF_ORG}/forged-fingerprints-contrast', repo_type='dataset', local_dir='data/contrast_sets/public')

In [ ]:
!python scripts/compute_fingerprint.py --config configs/fingerprint.yaml --split A
!python scripts/compute_fingerprint.py --config configs/fingerprint.yaml --split B --compare outputs/fingerprints/tiny-aya-global_A.pt

## Per-layer view

In [ ]:
from forged.fingerprint import Fingerprint, similarity
a = Fingerprint.load('outputs/fingerprints/tiny-aya-global_A.pt')
b = Fingerprint.load('outputs/fingerprints/tiny-aya-global_B.pt')
sim = similarity(a, b)
print('aggregate cosine:', round(sim['cosine'], 4))
for l, c in sim['per_layer'].items(): print(f'layer {l:2d}: {c:.4f}')

## Push the fingerprints (private results repo)

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
results_repo = f'{HF_ORG}/forged-fingerprints-results'
api.create_repo(results_repo, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path='outputs/fingerprints', path_in_repo='fingerprints', repo_id=results_repo,
                  repo_type='dataset', commit_message=f"Reference fingerprint, A vs B cosine {sim['cosine']:.4f}")
print('Pushed to', results_repo)